# SSIF_V3：Google Colab 中文實作教學（CSV streaming 修正版）

研究資料：

```python
data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
```

`combined_data.csv` 是「每列一個完整事件」的 CSV。`eq_info`、`intensity`、`stids`、`times` 與 `epicenter_distance` 都是巢狀 Python/JSON 字串；其中 `intensity` 是 `{station_id: [120 秒震度序列]}`。

本版不再使用 pandas chunk 將完整 CSV 轉換，而改用逐列 streaming converter，先掃描全檔並產生錯誤／重複事件報告，再轉為 SSIF event JSON。完整轉換失敗時，真正的 traceback 會直接顯示，並另存 `conversion_error_report.json`。


## 1. 掛載 Google Drive 並取得最新版程式


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import subprocess

REPO_ROOT = Path('/content/SSIF_V3')
subprocess.run(['rm', '-rf', str(REPO_ROOT)], check=True)
subprocess.run([
    'git', 'clone', 'https://github.com/oceanicdayi/SSIF_V3.git', str(REPO_ROOT)
], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True)
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'
], cwd=REPO_ROOT, check=True)


## 2. 設定資料路徑、輸出路徑與執行開關


In [ ]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import pandas as pd
import torch
from IPython.display import display

data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
DATA_CSV = Path(data_path)
WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')

TRAIN_DATA = WORK_ROOT / 'data' / 'training_archive_json'
EXTERNAL_DATA = WORK_ROOT / 'data' / 'external_evaluation_json'
REPORT_DIR = WORK_ROOT / 'reports'
PREPARED_DIR = WORK_ROOT / 'prepared' / 'split_v1'
MODEL_DIR = WORK_ROOT / 'models' / 'seed_20260728'
INFERENCE_DIR = WORK_ROOT / 'inference' / 'external_seed_20260728'
REPLAY_DIR = WORK_ROOT / 'replay'

SCAN_REPORT = REPORT_DIR / 'combined_data_scan.json'
CONVERSION_REPORT = TRAIN_DATA / 'conversion_summary.json'
VALIDATION_REPORT = REPORT_DIR / 'converted_archive_validation.json'

WINDOWS = [10, 15, 20, 25, 30, 35, 40]
SEED = 20260728

# 全檔 scan 是唯讀檢查；建議保持 True。
RUN_FULL_SCAN = True
# 確認 scan 沒有 row_errors 後再改為 True。
RUN_FULL_CONVERSION = False
RUN_FULL_VALIDATION = False
RUN_AUDIT_SPLIT = False
RUN_QUICK_TRAIN = False
RUN_FULL_TRAIN = False
RUN_EXTERNAL_EVAL = False

# exact duplicate 會略過；不同內容但相同事件身分會停止並要求人工審查。
DUPLICATE_POLICY = 'skip-identical'  # error / skip-identical / merge / suffix

for path in [TRAIN_DATA, EXTERNAL_DATA, REPORT_DIR, PREPARED_DIR, MODEL_DIR, INFERENCE_DIR, REPLAY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

assert DATA_CSV.is_file(), f'找不到研究資料：{DATA_CSV}'
print('CSV:', DATA_CSV)
print('CSV size (GB):', round(DATA_CSV.stat().st_size / 1024**3, 3))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. 先執行程式內建 smoke test


In [ ]:
subprocess.run([
    'python', 'smoke_test_combined_csv_conversion.py'
], cwd=REPO_ROOT, check=True)


## 4. 使用真實 CSV 前兩列檢查 schema


In [ ]:
inspect_cmd = [
    'python', 'combined_csv_to_ssif_json.py', 'inspect',
    '--csv', str(DATA_CSV),
    '--rows', '2',
    '--horizon', '120',
]
inspect_result = subprocess.run(
    inspect_cmd,
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(inspect_result.stdout)
if inspect_result.stderr:
    print(inspect_result.stderr)
assert inspect_result.returncode == 0, 'inspect 失敗，請查看上方 stderr'
inspect_summary = json.loads(inspect_result.stdout)
assert inspect_summary['layout'] == 'event_json'
assert {'eq_info', 'intensity'}.issubset(inspect_summary['columns'])
print('PASS: real CSV schema and first two rows')


## 5. 真實 CSV 前兩列端到端轉換


In [ ]:
SAMPLE_CSV = Path('/content/combined_data_sample_2rows.csv')
SAMPLE_OUT = Path('/content/ssif_combined_sample_events')
shutil.rmtree(SAMPLE_OUT, ignore_errors=True)

with DATA_CSV.open('r', encoding='utf-8-sig', newline='') as source:
    reader = csv.DictReader(source)
    assert reader.fieldnames is not None
    rows = []
    for index, row in enumerate(reader):
        rows.append(row)
        if index >= 1:
            break

with SAMPLE_CSV.open('w', encoding='utf-8', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=reader.fieldnames)
    writer.writeheader()
    writer.writerows(rows)

sample_convert = subprocess.run([
    'python', 'combined_csv_to_ssif_json.py', 'convert',
    '--csv', str(SAMPLE_CSV),
    '--output-dir', str(SAMPLE_OUT),
    '--horizon', '120',
    '--duplicate-policy', 'skip-identical',
    '--error-policy', 'error',
    '--overwrite',
], cwd=REPO_ROOT, text=True)
assert sample_convert.returncode == 0, '兩列轉換失敗；上方已顯示真正 traceback'

sample_validate = subprocess.run([
    'python', 'combined_csv_to_ssif_json.py', 'validate',
    '--data-dir', str(SAMPLE_OUT),
    '--horizon', '120',
], cwd=REPO_ROOT, text=True)
assert sample_validate.returncode == 0

sample_files = sorted(SAMPLE_OUT.glob('event_*.json'))
assert len(sample_files) == 2
sample_event = json.loads(sample_files[0].read_text(encoding='utf-8'))
first_station = next(iter(sample_event['intensity']))
assert len(sample_event['intensity'][first_station]) == 120
print('PASS: two real rows converted and validated')
print('Example event:', sample_files[0].name)
print('Stations:', len(sample_event['intensity']))
print('First station:', first_station)


## 6. 掃描完整 CSV（不寫 event JSON）

這一步會逐列解析整個 CSV，檢查：

- 無法解析的 `eq_info`、`intensity`、`stids` 或其他巢狀欄位
- 短於／長於 120 秒的測站序列
- exact duplicate rows
- 相同事件身分但內容不同的 identity conflicts
- 重複 origin time

報告保存到 `combined_data_scan.json`。轉換前必須先確認 `row_errors == 0`。


In [ ]:
if RUN_FULL_SCAN:
    scan_cmd = [
        'python', 'combined_csv_to_ssif_json.py', 'scan',
        '--csv', str(DATA_CSV),
        '--horizon', '120',
        '--max-errors', '50',
        '--report', str(SCAN_REPORT),
    ]
    scan_process = subprocess.run(scan_cmd, cwd=REPO_ROOT, text=True)
    assert scan_process.returncode == 0, 'scan 程式失敗；請查看上方 traceback'
else:
    print('RUN_FULL_SCAN=False：略過完整 scan')

if SCAN_REPORT.is_file():
    scan_summary = json.loads(SCAN_REPORT.read_text(encoding='utf-8'))
    counters = scan_summary['counters']
    display(pd.DataFrame([counters]).T.rename(columns={0: 'count'}))
    if scan_summary['errors']:
        display(pd.DataFrame(scan_summary['errors']))
    if scan_summary['duplicate_examples']:
        display(pd.DataFrame(scan_summary['duplicate_examples']))
    print('duplicate origin groups:', scan_summary['n_duplicate_origin_groups'])
    print('scan report:', SCAN_REPORT)
else:
    scan_summary = None


### Scan 判讀規則

- `row_errors > 0`：先修資料或解析器，不可正式轉換。
- `exact_duplicate_rows > 0` 且 `identity_conflicts == 0`：可使用 `skip-identical`。
- `identity_conflicts > 0`：先查看 `duplicate_examples`。`merge` 只會合併新增測站或完全相同的測站序列；同站序列衝突仍會停止。
- `suffix` 只適合確認兩列確實是不同事件時使用，不應用來隱藏重複資料。


In [ ]:
if scan_summary is not None:
    counters = scan_summary['counters']
    row_errors = counters.get('row_errors', 0)
    exact_duplicates = counters.get('exact_duplicate_rows', 0)
    identity_conflicts = counters.get('identity_conflicts', 0)
    print('row_errors:', row_errors)
    print('exact_duplicate_rows:', exact_duplicates)
    print('identity_conflicts:', identity_conflicts)
    if row_errors:
        print('STOP: 請先查看 scan_summary["errors"]。')
    elif identity_conflicts and DUPLICATE_POLICY == 'skip-identical':
        print('STOP: 發現不同內容的重複事件；請審查後選擇 merge、suffix 或修正來源資料。')
    else:
        print('PASS: scan result is compatible with current duplicate policy')


## 7. 完整 streaming 轉換


In [ ]:
if RUN_FULL_CONVERSION:
    assert SCAN_REPORT.is_file(), '請先執行完整 scan'
    scan_summary = json.loads(SCAN_REPORT.read_text(encoding='utf-8'))
    counters = scan_summary['counters']
    assert counters.get('row_errors', 0) == 0, 'scan 有 row_errors，禁止正式轉換'
    if counters.get('identity_conflicts', 0) > 0:
        assert DUPLICATE_POLICY in {'merge', 'suffix'}, (
            '存在 identity_conflicts；先審查 duplicate_examples，再設定 merge 或 suffix'
        )

    full_convert_cmd = [
        'python', 'combined_csv_to_ssif_json.py', 'convert',
        '--csv', str(DATA_CSV),
        '--output-dir', str(TRAIN_DATA),
        '--horizon', '120',
        '--duplicate-policy', DUPLICATE_POLICY,
        '--error-policy', 'error',
        '--max-errors', '50',
        '--overwrite',
    ]
    # 不使用 capture_output，完整 traceback 會直接顯示在 Colab。
    completed = subprocess.run(full_convert_cmd, cwd=REPO_ROOT, text=True)
    if completed.returncode != 0:
        error_report = TRAIN_DATA / 'conversion_error_report.json'
        if error_report.is_file():
            print(error_report.read_text(encoding='utf-8'))
        raise RuntimeError('完整 CSV 轉換失敗；請查看上方 traceback 與 error report')
    conversion_summary = json.loads(CONVERSION_REPORT.read_text(encoding='utf-8'))
    print(json.dumps(conversion_summary, ensure_ascii=False, indent=2))
else:
    print('RUN_FULL_CONVERSION=False：scan 確認後再開啟')


## 8. 驗證完整轉換結果


In [ ]:
if RUN_FULL_VALIDATION:
    validate_cmd = [
        'python', 'combined_csv_to_ssif_json.py', 'validate',
        '--data-dir', str(TRAIN_DATA),
        '--horizon', '120',
        '--max-errors', '50',
        '--report', str(VALIDATION_REPORT),
    ]
    validation_process = subprocess.run(validate_cmd, cwd=REPO_ROOT, text=True)
    assert validation_process.returncode == 0
    archive_validation = json.loads(VALIDATION_REPORT.read_text(encoding='utf-8'))
    assert archive_validation['counters'].get('errors', 0) == 0
    assert archive_validation['counters'].get('event_json', 0) > 0

    conversion_summary = json.loads(CONVERSION_REPORT.read_text(encoding='utf-8'))
    assert archive_validation['counters']['event_json'] == conversion_summary['n_events']
    print('PASS: converted archive is structurally valid')
else:
    print('RUN_FULL_VALIDATION=False')


## 9. SSIF loader 相容性檢查


In [ ]:
if RUN_FULL_VALIDATION:
    import sys
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    from ssif_core import load_station_records

    records, loader_stats = load_station_records(
        TRAIN_DATA,
        label_horizon=120,
        require_label_horizon=True,
        min_full_valid_fraction=0.80,
    )
    print(json.dumps(loader_stats, ensure_ascii=False, indent=2))
    assert records
    assert all(record.label_horizon == 120 for record in records)
    print('PASS: SSIF loader accepted the converted archive')


## 10. 資料稽核與事件層級四集合切分


In [ ]:
if RUN_AUDIT_SPLIT:
    assert TRAIN_DATA.exists() and any(TRAIN_DATA.glob('event_*.json'))
    if PREPARED_DIR.exists() and any(PREPARED_DIR.iterdir()):
        raise RuntimeError(
            f'{PREPARED_DIR} 已有輸出。資料版本改變時請使用新的 split_v2，避免覆蓋研究紀錄。'
        )
    audit_cmd = [
        'python', 'prepare_ssif_dataset.py', 'audit-split',
        '--data-dir', str(TRAIN_DATA),
        '--output-dir', str(PREPARED_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--min-label-valid-fraction', '0.80',
        '--min-window-valid-fraction', '0.80',
        '--train-ratio', '0.70',
        '--validation-ratio', '0.10',
        '--calibration-ratio', '0.10',
        '--test-ratio', '0.10',
        '--split-candidates', '5000',
        '--seed', str(SEED),
    ]
    subprocess.run(audit_cmd, cwd=REPO_ROOT, check=True)
else:
    print('RUN_AUDIT_SPLIT=False')


## 11. 檢查 audit 與 split 結果


In [ ]:
if (PREPARED_DIR / 'audit_summary.json').is_file():
    audit_summary = json.loads((PREPARED_DIR / 'audit_summary.json').read_text(encoding='utf-8'))
    print(json.dumps(audit_summary, ensure_ascii=False, indent=2))
    event_split = pd.read_csv(PREPARED_DIR / 'event_split.csv')
    event_audit = pd.read_csv(PREPARED_DIR / 'event_audit.csv')
    display(event_split['split'].value_counts().rename_axis('split').to_frame('events'))
    display(event_audit.head())


## 12. EW10 一個 epoch 快速訓練


In [ ]:
QUICK_MODEL_DIR = WORK_ROOT / 'models' / 'quick_EW10'
if RUN_QUICK_TRAIN:
    quick_cmd = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(QUICK_MODEL_DIR),
        '--windows', '10',
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '1',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--workers', '2',
    ]
    if torch.cuda.is_available():
        quick_cmd.append('--amp')
    subprocess.run(quick_cmd, cwd=REPO_ROOT, check=True)
    assert (QUICK_MODEL_DIR / 'EW10' / 'best.pt').is_file()
    print('PASS: EW10 quick training')
else:
    print('RUN_QUICK_TRAIN=False')


## 13. 正式訓練 EW10–EW40


In [ ]:
if RUN_FULL_TRAIN:
    train_cmd = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(MODEL_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '30',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--weight-decay', '1e-2',
        '--warmup-ratio', '0.10',
        '--min-precision', '0.90',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--patience', '6',
        '--workers', '2',
    ]
    if torch.cuda.is_available():
        train_cmd.append('--amp')
    subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)
else:
    print('RUN_FULL_TRAIN=False')


## 14. 查看訓練摘要


In [ ]:
if (MODEL_DIR / 'summary.json').is_file():
    training_summary = json.loads((MODEL_DIR / 'summary.json').read_text(encoding='utf-8'))
    rows = []
    for item in training_summary:
        alert = item['test']['alert']
        rows.append({
            'window': item['window'],
            'best_epoch': item['best_epoch'],
            'threshold': item['threshold'],
            'precision': alert['precision'],
            'pod': alert['pod'],
            'f1': alert['f1'],
            'fpr': alert['fpr'],
        })
    display(pd.DataFrame(rows).sort_values('window'))


## 15. 獨立資料 inference


In [ ]:
if RUN_EXTERNAL_EVAL:
    external_files = sorted(EXTERNAL_DATA.glob('event_*.json'))
    assert external_files, 'EXTERNAL_DATA 沒有獨立事件 JSON'
    eval_cmd = [
        'python', 'train_ssif_v3.py', 'evaluate-all',
        '--data-dir', str(EXTERNAL_DATA),
        '--model-root', str(MODEL_DIR),
        '--output-dir', str(INFERENCE_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--cohort', 'common',
        '--batch-size', '128',
        '--workers', '2',
    ]
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
else:
    print('RUN_EXTERNAL_EVAL=False')


## 執行順序

1. 先執行到第 5 節，確認 smoke test、inspect 與兩列真實資料轉換都 PASS。
2. 第 6 節完整 scan 完成後，檢查 `row_errors`、`exact_duplicate_rows` 與 `identity_conflicts`。
3. 只有 `row_errors == 0` 才將 `RUN_FULL_CONVERSION=True`。
4. 完整轉換後將 `RUN_FULL_VALIDATION=True`，並確認 SSIF loader 可讀。
5. 再依序開啟 audit、EW10 quick training 與完整訓練。

不要把同一 `combined_data.csv` 複製成 external evaluation。外部評估必須使用未參與訓練、validation、calibration 與模型選擇的獨立事件。
